In [32]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import OneCycleLR
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")

Device: cuda
GPU: Tesla T4


In [36]:
import os

# Check karo kya hai /kaggle/input mein
print("Available datasets:")
for folder in os.listdir('/kaggle/input'):
    print(f"  /kaggle/input/{folder}/")
    for f in os.listdir(f'/kaggle/input/{folder}')[:5]:
        print(f"    └── {f}")

Available datasets:
  /kaggle/input/competitions/
    └── isic-2024-challenge


In [39]:
import os

# ── Sahi path ──────────────────────────────────────────────────
BASE    = '/kaggle/input/competitions/isic-2024-challenge'
IMG_DIR = f'{BASE}/train-image/image'

# ── Verify karo pehle ──────────────────────────────────────────
print("Files in BASE:")
for f in os.listdir(BASE):
    print(f"  └── {f}")

Files in BASE:
  └── sample_submission.csv
  └── train-metadata.csv
  └── test-metadata.csv
  └── test-image.hdf5
  └── train-image
  └── train-image.hdf5


In [40]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(f'{BASE}/train-metadata.csv')
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nClass distribution:")
print(df['target'].value_counts())

df['age_approx'] = df['age_approx'].fillna(df['age_approx'].median())
scaler = StandardScaler()
df['age_scaled'] = scaler.fit_transform(df[['age_approx']])
df = pd.get_dummies(df, columns=['sex', 'anatom_site_general'],
                    drop_first=False, dtype=float)
print("\nData clean ho gaya ✓")

/tmp/ipykernel_55/1112940414.py:4: DtypeWarning: Columns (51,52) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f'{BASE}/train-metadata.csv')


Shape: (401059, 55)
Columns: ['isic_id', 'target', 'patient_id', 'age_approx', 'sex', 'anatom_site_general', 'clin_size_long_diam_mm', 'image_type', 'tbp_tile_type', 'tbp_lv_A', 'tbp_lv_Aext', 'tbp_lv_B', 'tbp_lv_Bext', 'tbp_lv_C', 'tbp_lv_Cext', 'tbp_lv_H', 'tbp_lv_Hext', 'tbp_lv_L', 'tbp_lv_Lext', 'tbp_lv_areaMM2', 'tbp_lv_area_perim_ratio', 'tbp_lv_color_std_mean', 'tbp_lv_deltaA', 'tbp_lv_deltaB', 'tbp_lv_deltaL', 'tbp_lv_deltaLB', 'tbp_lv_deltaLBnorm', 'tbp_lv_eccentricity', 'tbp_lv_location', 'tbp_lv_location_simple', 'tbp_lv_minorAxisMM', 'tbp_lv_nevi_confidence', 'tbp_lv_norm_border', 'tbp_lv_norm_color', 'tbp_lv_perimeterMM', 'tbp_lv_radial_color_std_max', 'tbp_lv_stdL', 'tbp_lv_stdLExt', 'tbp_lv_symm_2axis', 'tbp_lv_symm_2axis_angle', 'tbp_lv_x', 'tbp_lv_y', 'tbp_lv_z', 'attribution', 'copyright_license', 'lesion_id', 'iddx_full', 'iddx_1', 'iddx_2', 'iddx_3', 'iddx_4', 'iddx_5', 'mel_mitotic_index', 'mel_thick_mm', 'tbp_lv_dnn_lesion_confidence']

Class distribution:
target


In [41]:
META_COLS = [c for c in df.columns
             if c.startswith('sex_') or
                c.startswith('anatom_site_general_') or
                c == 'age_scaled']
print(f"Meta columns: {META_COLS}")

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
fold_splits = list(sgkf.split(df, df['target'], groups=df['patient_id']))
train_idx, val_idx = fold_splits[0]

df_train = df.iloc[train_idx].reset_index(drop=True)
df_val   = df.iloc[val_idx].reset_index(drop=True)

print(f"Train: {len(df_train):,} | Val: {len(df_val):,}")
print(f"Train cancer: {df_train['target'].sum()} ({df_train['target'].mean():.2%})")
print(f"Val cancer  : {df_val['target'].sum()} ({df_val['target'].mean():.2%})")

Meta columns: ['age_scaled', 'sex_female', 'sex_male', 'anatom_site_general_anterior torso', 'anatom_site_general_head/neck', 'anatom_site_general_lower extremity', 'anatom_site_general_posterior torso', 'anatom_site_general_upper extremity']
Train: 329,895 | Val: 71,164
Train cancer: 310 (0.09%)
Val cancer  : 83 (0.12%)


In [42]:
class ISIC2024Dataset(Dataset):
    def __init__(self, df, img_dir, meta_cols, transform=None):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.meta_cols = meta_cols
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = f"{self.img_dir}/{row['isic_id']}.jpg"
        img      = Image.open(img_path).convert('RGB')

        if self.transform:
            img = self.transform(img)

        meta  = torch.tensor(row[self.meta_cols].values.astype('float32'))
        label = torch.tensor(row['target'], dtype=torch.float32)
        return img, meta, label

print("Dataset class ready ✓")

Dataset class ready ✓


In [43]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(90),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_ds = ISIC2024Dataset(df_train, IMG_DIR, META_COLS, train_transform)
val_ds   = ISIC2024Dataset(df_val,   IMG_DIR, META_COLS, val_transform)

labels   = df_train['target'].values.astype(int)
class_w  = 1.0 / np.bincount(labels)
sample_w = class_w[labels]
sampler  = WeightedRandomSampler(
    weights=torch.FloatTensor(sample_w),
    num_samples=len(sample_w),
    replacement=True
)

train_loader = DataLoader(train_ds, batch_size=64, sampler=sampler,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")
print("DataLoaders ready ✓")

Train batches: 5155
Val batches  : 1112
DataLoaders ready ✓


In [44]:
class LesionAttentionGate(nn.Module):
    def __init__(self, feat_channels=2048, meta_dim=16):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(meta_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, feat_channels),
            nn.Sigmoid()
        )
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, feat_map, meta):
        gate  = self.gate(meta).unsqueeze(-1).unsqueeze(-1)
        gated = feat_map * gate
        return self.pool(gated).flatten(1)


class FusionSkinNet(nn.Module):
    def __init__(self, meta_dim):
        super().__init__()
        base = models.resnet50(weights='IMAGENET1K_V2')

        for name, param in base.named_parameters():
            param.requires_grad = any(
                layer in name for layer in ['layer3', 'layer4', 'fc']
            )

        self.encoder = nn.Sequential(*list(base.children())[:-2])
        self.lag     = LesionAttentionGate(2048, meta_dim)
        self.head    = nn.Sequential(
            nn.Linear(2048, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )
        nn.init.kaiming_normal_(self.head[0].weight, mode='fan_out')

    def forward(self, img, meta):
        feat = self.encoder(img)
        att  = self.lag(feat, meta)
        return self.head(att)


meta_dim  = len(META_COLS)
model     = FusionSkinNet(meta_dim).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {trainable:,}")
print("Model ready ✓")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 188MB/s] 


Trainable params: 24,235,521
Model ready ✓


In [45]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce  = nn.functional.binary_cross_entropy_with_logits(
                   logits, targets, reduction='none')
        pt   = torch.exp(-bce)
        loss = self.alpha * (1 - pt)**self.gamma * bce
        return loss.mean()

criterion = FocalLoss(alpha=0.25, gamma=2.0)

optimizer = torch.optim.AdamW([
    {'params': model.encoder.parameters(), 'lr': 5e-5},
    {'params': model.lag.parameters(),     'lr': 5e-4},
    {'params': model.head.parameters(),    'lr': 5e-4},
], weight_decay=1e-4)

print("Loss + Optimizer ready ✓")

Loss + Optimizer ready ✓


In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0

    for imgs, metas, labels in tqdm(loader, desc='Training', leave=False):
        imgs, metas, labels = imgs.to(device), metas.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs, metas).squeeze(1)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader, device, threshold=0.3):
    model.eval()
    all_probs, all_labels = [], []

    with torch.no_grad():
        for imgs, metas, labels in tqdm(loader, desc='Evaluating', leave=False):
            probs = torch.sigmoid(
                model(imgs.to(device), metas.to(device)).squeeze(1)
            ).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())

    probs  = np.array(all_probs)
    labels = np.array(all_labels)
    preds  = (probs >= threshold).astype(int)

    try:
        pauc = roc_auc_score(labels, probs, max_fpr=0.2)
    except:
        pauc = 0.0

    return {
        'auc'   : roc_auc_score(labels, probs),
        'pauc'  : pauc,
        'recall': recall_score(labels, preds, zero_division=0),
        'f1'    : f1_score(labels, preds, zero_division=0),
        'probs' : probs,
        'labels': labels,
    }

print("Functions ready ✓")

In [ ]:
from torch.optim.lr_scheduler import OneCycleLR

# ── Sirf EPOCHS aur scheduler change kiya ─────────────────────
EPOCHS    = 6          # 15 → 6 kar diya
scheduler = OneCycleLR(
    optimizer,
    max_lr          = [5e-5, 5e-4, 5e-4],
    steps_per_epoch = len(train_loader),
    epochs          = EPOCHS,   # yahan bhi 6 ho gaya
    pct_start       = 0.1
)

best_pauc = 0.0
best_path = '/kaggle/working/best_model.pth'
history   = []

print("=" * 50)
print("  TRAINING SHURU!")
print(f"  Epochs: {EPOCHS} | Batches/epoch: {len(train_loader)}")
print("=" * 50)

for epoch in range(1, EPOCHS + 1):
    train_loss  = train_one_epoch(model, train_loader, optimizer,
                                  scheduler, criterion, device)
    val_metrics = evaluate(model, val_loader, device, threshold=0.3)

    history.append({
        'epoch' : epoch,
        'loss'  : train_loss,
        'auc'   : val_metrics['auc'],
        'pauc'  : val_metrics['pauc'],
        'recall': val_metrics['recall'],
        'f1'    : val_metrics['f1'],
    })

    print(f"\nEpoch {epoch:02d}/{EPOCHS}")
    print(f"  Loss  : {train_loss:.4f}")
    print(f"  AUC   : {val_metrics['auc']:.4f}")
    print(f"  pAUC  : {val_metrics['pauc']:.4f}  <- main metric")
    print(f"  Recall: {val_metrics['recall']:.4f}")
    print(f"  F1    : {val_metrics['f1']:.4f}")

    if val_metrics['pauc'] > best_pauc:
        best_pauc = val_metrics['pauc']
        torch.save(model.state_dict(), best_path)
        print(f"  BEST MODEL SAVE! pAUC = {best_pauc:.4f}")
s
print(f"\nBest pAUC: {best_pauc:.4f}")

In [50]:
# ── Load best model from Epoch 2 ─────────────────────────────
model.load_state_dict(torch.load('/kaggle/working/best_model.pth'))

# ── LOWER learning rates — half karo ─────────────────────────
optimizer = torch.optim.AdamW([
    {'params': model.encoder.parameters(), 'lr': 2.5e-5},  # 5e-5 → 2.5e-5
    {'params': model.lag.parameters(),     'lr': 2.5e-4},  # 5e-4 → 2.5e-4
    {'params': model.head.parameters(),    'lr': 2.5e-4},  # 5e-4 → 2.5e-4
], weight_decay=1e-4)

EPOCHS    = 2  # sirf 2 aur epochs (total 6 = 2+4 pehle wale)
scheduler = OneCycleLR(
    optimizer,
    max_lr          = [2.5e-5, 2.5e-4, 2.5e-4],  # update yahan bhi
    steps_per_epoch = len(train_loader),
    epochs          = EPOCHS,
    pct_start       = 0.1
)

best_pauc = 0.8158  # Epoch 2 ka value
best_path = '/kaggle/working/best_model.pth'

print("=" * 50)
print("  RESUMING with LOWER LR (from Epoch 2)")
print("=" * 50)

for epoch in range(5, 7):  # epochs 5-6
    train_loss  = train_one_epoch(model, train_loader, optimizer,
                                  scheduler, criterion, device)
    val_metrics = evaluate(model, val_loader, device, threshold=0.3)
    
    history.append({
        'epoch' : epoch,
        'loss'  : train_loss,
        'auc'   : val_metrics['auc'],
        'pauc'  : val_metrics['pauc'],
        'recall': val_metrics['recall'],
        'f1'    : val_metrics['f1'],
    })

    print(f"\nEpoch {epoch:02d}/6")
    print(f"  Loss  : {train_loss:.4f}")
    print(f"  AUC   : {val_metrics['auc']:.4f}")
    print(f"  pAUC  : {val_metrics['pauc']:.4f}  <- main metric")
    print(f"  Recall: {val_metrics['recall']:.4f}")
    print(f"  F1    : {val_metrics['f1']:.4f}")

    if val_metrics['pauc'] > best_pauc:
        best_pauc = val_metrics['pauc']
        torch.save(model.state_dict(), best_path)
        print(f"  ✅ NEW BEST! pAUC = {best_pauc:.4f}")
    else:
        print(f"  (Best so far: {best_pauc:.4f})")

print(f"\n{'='*50}")
print(f"  FINAL BEST pAUC: {best_pauc:.4f}")
print(f"{'='*50}")

  RESUMING with LOWER LR (from Epoch 2)


Training:   0%|          | 0/5155 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/1112 [00:00<?, ?it/s]


Epoch 05/6
  Loss  : 0.0004
  AUC   : 0.8828
  pAUC  : 0.7924  <- main metric
  Recall: 0.1325
  F1    : 0.1028
  (Best so far: 0.8158)


Training:   0%|          | 0/5155 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/1112 [00:00<?, ?it/s]


Epoch 06/6
  Loss  : 0.0002
  AUC   : 0.8536
  pAUC  : 0.7579  <- main metric
  Recall: 0.1205
  F1    : 0.0794
  (Best so far: 0.8158)

  FINAL BEST pAUC: 0.8158


In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # ← No display backend, much faster
import matplotlib.pyplot as plt
import numpy as np

# ── History se directly final results ────────────────────────
hist_df = pd.DataFrame(history)
best_row = hist_df[hist_df['epoch'] == 2].iloc[0]

print("=" * 60)
print("  FINAL RESULTS (BEST MODEL - EPOCH 2)")
print("=" * 60)
print(f"  Loss  : {best_row['loss']:.4f}")
print(f"  AUC   : {best_row['auc']:.4f}")
print(f"  pAUC  : {best_row['pauc']:.4f}  ⭐ ISIC 2024 official metric")
print(f"  Recall: {best_row['recall']:.4f}")
print(f"  F1    : {best_row['f1']:.4f}")
print("=" * 60)

# ── Learning curves ──────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle('ISIC 2024 — Training Progress (6 Epochs)', fontsize=16, fontweight='bold')

for ax, col, color, title in zip(
    axes,
    ['loss', 'auc', 'pauc', 'recall'],
    ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A'],
    ['Train Loss', 'Val AUC', 'Val pAUC (Main Metric)', 'Val Recall']
):
    ax.plot(hist_df['epoch'], hist_df[col], marker='o', color=color, linewidth=2.5, markersize=8)
    ax.axvline(x=2, color='red', linestyle='--', alpha=0.5, linewidth=2, label='Best Model')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel(col.upper(), fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('/kaggle/working/learning_curves.png', dpi=100, bbox_inches='tight')  # ← dpi 200→100
plt.close()  # ← replaces plt.show(), instantly frees memory

print("✓ learning_curves.png saved")

# ── Save history ─────────────────────────────────────────────
hist_df.to_csv('/kaggle/working/training_history.csv', index=False)

print("\n" + "=" * 60)
print("  FILES SAVED")
print("=" * 60)
print("  ✓ learning_curves.png")
print("  ✓ training_history.csv")
print("  ✓ best_model.pth (pAUC: 0.8158)")
print("\nReady for PowerPoint! 🎉")
print("=" * 60)